# TFM - EfficientNet-B2 Transfer Learning progresivo para clasificación de defectos (NEU-DET)

Este notebook replica la estructura de `02_cnn_entrenamiento.ipynb` y entrena un modelo **EfficientNet-B2 preentrenado** en 3 fases de fine-tuning progresivo para las 6 clases de NEU-DET: `crazing`, `inclusion`, `patches`, `pitted_surface`, `rolled-in_scale` y `scratches`.

Fases:
1. Entrenar solo `classifier` con backbone congelado (LR=1e-3).
2. Descongelar `features.8` + `classifier` (LR=1e-5).
3. Descongelar `features.7` + `features.8` + `classifier` (LR=5e-6).

En cada fase se generan curvas, matriz de confusión, métricas por clase y muestras clasificadas, además de evaluación en TEST externo.

In [ ]:
# ============================
# 1) Importaciones y configuración
# ============================
import copy
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

try:
    import albumentations as A
except ImportError:
    %pip install -q albumentations opencv-python-headless
    import albumentations as A

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo en uso: {device}')

In [ ]:
# ==========================================
# 2) Transformaciones y carga del dataset
# ==========================================
data_dir = Path('../data/clasificacion_full')

class AlbumentationsAugment:
    """Aplica GridDistortion + Cutout sobre imagen PIL y retorna PIL."""

    def __init__(self):
        self.transform = A.Compose([
            A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.5),
            A.CoarseDropout(
                num_holes_range=(1, 4),
                hole_height_range=(16, 48),
                hole_width_range=(16, 48),
                fill=0,
                p=0.5
            )
        ])

    def __call__(self, img):
        img_np = np.array(img)
        augmented = self.transform(image=img_np)['image']
        return Image.fromarray(augmented)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    AlbumentationsAugment(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.12), ratio=(0.3, 3.3), value='random')
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

base_dataset = datasets.ImageFolder(root=data_dir)
class_names = base_dataset.classes
num_classes = len(class_names)

print(f'Total de imágenes: {len(base_dataset)}')
print(f'Número de clases: {num_classes}')
print(f'Clases detectadas: {class_names}')

In [ ]:
# ==================================================
# 3) División Train/Validation y DataLoaders
# ==================================================
targets = np.array(base_dataset.targets)
indices = np.arange(len(base_dataset))

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.2,
    random_state=SEED,
    stratify=targets
)

train_dataset_full = datasets.ImageFolder(root=data_dir, transform=train_transform)
val_dataset_full = datasets.ImageFolder(root=data_dir, transform=val_transform)

train_dataset = Subset(train_dataset_full, train_indices)
val_dataset = Subset(val_dataset_full, val_indices)

batch_size = 32
num_workers = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available()
)

print(f'Muestras de entrenamiento: {len(train_dataset)}')
print(f'Muestras de validación: {len(val_dataset)}')

In [ ]:
# ==================================================
# 4) Funciones auxiliares del modelo y entrenamiento
# ==================================================
def create_model(num_classes=6):
    """Crea EfficientNet-B2 preentrenada y adapta su clasificador."""
    if hasattr(models, 'EfficientNet_B2_Weights'):
        model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)
    else:
        model = models.efficientnet_b2(pretrained=True)

    in_features = model.classifier[1].in_features  # EfficientNet-B2: 1408
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.4),
        nn.Linear(512, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_classes)
    )
    return model

def freeze_backbone(model):
    """Congela todas las capas del backbone y deja classifier entrenable."""
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

def unfreeze_layers(model, layer_names):
    """Descongela bloques de features por nombre y mantiene classifier entrenable."""
    for param in model.features.parameters():
        param.requires_grad = False

    for layer_name in layer_names:
        layer = model.features[int(layer_name.split('.')[-1])]
        for param in layer.parameters():
            param.requires_grad = True

    for param in model.classifier.parameters():
        param.requires_grad = True

def train_one_epoch(model, dataloader, criterion, optimizer, device='cpu'):
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total_samples = 0
    all_preds = []
    all_labels = []

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs, dim=1)

        batch_n = labels.size(0)
        running_loss += loss.item() * batch_n
        running_corrects += (preds == labels).sum().item()
        total_samples += batch_n

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / total_samples
    epoch_acc = running_corrects / total_samples
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='macro', zero_division=0
    )
    return epoch_loss, epoch_acc, precision_macro, recall_macro, f1_macro

def validate(model, dataloader, criterion, device='cpu'):
    model.eval()
    running_loss = 0.0
    running_corrects = 0
    total_samples = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)
            preds = torch.argmax(outputs, dim=1)

            batch_n = labels.size(0)
            running_loss += loss.item() * batch_n
            running_corrects += (preds == labels).sum().item()
            total_samples += batch_n

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / total_samples
    epoch_acc = running_corrects / total_samples
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='macro', zero_division=0
    )

    return epoch_loss, epoch_acc, precision_macro, recall_macro, f1_macro, all_labels, all_preds

def plot_training_curves(history, save_dir, phase_name):
    save_dir.mkdir(parents=True, exist_ok=True)
    num_epochs_trained = len(history['train_loss'])
    epochs = np.arange(1, num_epochs_trained + 1)

    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], label='Train Loss', marker='o')
    plt.plot(epochs, history['val_loss'], label='Val Loss', marker='o')
    plt.title(f'Curvas de Pérdida - {phase_name}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(alpha=0.3)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], label='Train Accuracy', marker='o')
    plt.plot(epochs, history['val_acc'], label='Val Accuracy', marker='o')
    plt.title(f'Curvas de Precisión - {phase_name}')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    learning_curves_path = save_dir / f'curvas_aprendizaje_{phase_name}.png'
    plt.savefig(learning_curves_path, dpi=300, bbox_inches='tight')
    print(f'Gráfica guardada en: {learning_curves_path}')
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(epochs, history['lr'], label='Learning Rate', marker='o', color='tab:green')
    plt.title(f'Evolución del Learning Rate - {phase_name}')
    plt.xlabel('Epoch')
    plt.ylabel('LR')
    plt.yscale('log')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    lr_path = save_dir / f'evolucion_lr_{phase_name}.png'
    plt.savefig(lr_path, dpi=300, bbox_inches='tight')
    print(f'Gráfica guardada en: {lr_path}')
    plt.show()

    plt.figure(figsize=(14, 5))
    plt.subplot(1, 3, 1)
    plt.plot(epochs, history['train_precision_macro'], label='Train', marker='o')
    plt.plot(epochs, history['val_precision_macro'], label='Val', marker='o')
    plt.title(f'Precisión Macro - {phase_name}')
    plt.xlabel('Epoch')
    plt.ylabel('Precision')
    plt.grid(alpha=0.3)
    plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot(epochs, history['train_recall_macro'], label='Train', marker='o')
    plt.plot(epochs, history['val_recall_macro'], label='Val', marker='o')
    plt.title(f'Recall Macro - {phase_name}')
    plt.xlabel('Epoch')
    plt.ylabel('Recall')
    plt.grid(alpha=0.3)
    plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(epochs, history['train_f1_macro'], label='Train', marker='o')
    plt.plot(epochs, history['val_f1_macro'], label='Val', marker='o')
    plt.title(f'F1-Score Macro - {phase_name}')
    plt.xlabel('Epoch')
    plt.ylabel('F1-Score')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    metrics_path = save_dir / f'metricas_macro_{phase_name}.png'
    plt.savefig(metrics_path, dpi=300, bbox_inches='tight')
    print(f'Gráfica guardada en: {metrics_path}')
    plt.show()

def plot_confusion_matrix(labels, preds, class_names, save_dir, title_suffix):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.title(f'Matriz de Confusión - {title_suffix}')
    plt.xlabel('Predicción')
    plt.ylabel('Etiqueta Real')
    plt.tight_layout()

    out_path = save_dir / f"matriz_confusion_{title_suffix.lower().replace(' ', '_')}.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    print(f'Matriz de confusión guardada en: {out_path}')
    plt.show()

def show_sample_predictions(model, dataset, class_names, device, save_dir, title_suffix, n_samples=12):
    model.eval()
    sample_indices = np.random.choice(len(dataset), size=min(n_samples, len(dataset)), replace=False)

    fig = plt.figure(figsize=(16, 10))
    rows = int(np.ceil(len(sample_indices) / 4))

    with torch.no_grad():
        for i, idx in enumerate(sample_indices, start=1):
            image, true_label = dataset[idx]
            image_batch = image.unsqueeze(0).to(device)
            logits = model(image_batch)
            pred_label = torch.argmax(logits, dim=1).item()

            img_np = image.permute(1, 2, 0).cpu().numpy()
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            img_np = np.clip((img_np * std + mean), 0, 1)

            ax = fig.add_subplot(rows, 4, i)
            ax.imshow(img_np)
            color = 'green' if pred_label == true_label else 'red'
            ax.set_title(f'T: {class_names[true_label]}\nP: {class_names[pred_label]}', color=color, fontsize=9)
            ax.axis('off')

    fig.suptitle(f'Muestras clasificadas - {title_suffix}', fontsize=14)
    fig.tight_layout()
    out_path = save_dir / f"muestras_clasificadas_{title_suffix.lower().replace(' ', '_')}.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    print(f'Muestras guardadas en: {out_path}')
    plt.show()

In [ ]:
# ==================================================
# 5) Funciones auxiliares de evaluación y fases
# ==================================================
def evaluate_on_test(model, class_names, val_transform, batch_size, num_workers, device, phase_dir):
    test_dir = Path('../data/test')
    if not test_dir.exists():
        raise FileNotFoundError(f'No existe la carpeta de test: {test_dir.resolve()}')

    test_dataset = datasets.ImageFolder(root=str(test_dir), transform=val_transform)

    if set(test_dataset.classes) != set(class_names):
        raise ValueError(
            f'Clases distintas entre train y test. train={class_names}, test={test_dataset.classes}'
        )

    train_class_to_idx = {name: i for i, name in enumerate(class_names)}
    idx_map_test_to_train = {
        test_idx: train_class_to_idx[class_name]
        for class_name, test_idx in test_dataset.class_to_idx.items()
    }

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )

    model.eval()
    test_preds = []
    test_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()

            mapped_labels = [idx_map_test_to_train[int(l)] for l in labels.numpy()]
            test_preds.extend(preds)
            test_labels.extend(mapped_labels)

    test_acc = accuracy_score(test_labels, test_preds)
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
        test_labels, test_preds, average='macro', zero_division=0
    )

    print('Resultados en TEST externo:')
    print(f'Accuracy:  {test_acc:.4f}')
    print(f'Precision macro: {prec_macro:.4f}')
    print(f'Recall macro:    {rec_macro:.4f}')
    print(f'F1 macro:        {f1_macro:.4f}')

    plot_confusion_matrix(test_labels, test_preds, class_names, phase_dir, 'TEST externo')

    report_test = classification_report(
        test_labels,
        test_preds,
        target_names=class_names,
        digits=4,
    )
    print('Reporte de clasificación (TEST):')
    print(report_test)

    report_test_path = phase_dir / 'reporte_clasificacion_test.txt'
    with open(report_test_path, 'w', encoding='utf-8') as f:
        f.write(report_test)
    print(f'Reporte de test guardado en: {report_test_path}')

def train_phase(model, phase_name, num_epochs, lr, phase_dir):
    criterion = nn.CrossEntropyLoss()
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(trainable_params, lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=3,
        min_lr=1e-7
    )

    patience = 10
    patience_counter = 0
    best_val_acc = 0.0
    best_model_state = copy.deepcopy(model.state_dict())

    history = {
        'train_loss': [],
        'train_acc': [],
        'train_precision_macro': [],
        'train_recall_macro': [],
        'train_f1_macro': [],
        'val_loss': [],
        'val_acc': [],
        'val_precision_macro': [],
        'val_recall_macro': [],
        'val_f1_macro': [],
        'lr': []
    }

    for epoch in range(num_epochs):
        train_loss, train_acc, train_prec, train_rec, train_f1 = train_one_epoch(
            model=model,
            dataloader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device
        )

        val_loss, val_acc, val_prec, val_rec, val_f1, all_labels, all_preds = validate(
            model=model,
            dataloader=val_loader,
            criterion=criterion,
            device=device
        )

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_precision_macro'].append(train_prec)
        history['train_recall_macro'].append(train_rec)
        history['train_f1_macro'].append(train_f1)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_precision_macro'].append(val_prec)
        history['val_recall_macro'].append(val_rec)
        history['val_f1_macro'].append(val_f1)
        history['lr'].append(current_lr)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        print(
            f'{phase_name} | Epoch [{epoch + 1}/{num_epochs}] | '
            f'LR: {current_lr:.7f} | '
            f'Train [Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}] | '
            f'Val [Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}]'
        )

        if patience_counter >= patience:
            print(f'Early Stopping activado en {phase_name}, epoch {epoch + 1}/{num_epochs}.')
            break

    model.load_state_dict(best_model_state)
    print(f'Mejor Val Acc ({phase_name}): {best_val_acc:.4f}')

    plot_training_curves(history, phase_dir, phase_name)

    _, _, _, _, _, val_labels, val_preds = validate(
        model=model,
        dataloader=val_loader,
        criterion=criterion,
        device=device
    )

    plot_confusion_matrix(val_labels, val_preds, class_names, phase_dir, 'Validación')

    report = classification_report(val_labels, val_preds, target_names=class_names, digits=4)
    print('Reporte de clasificación (validación):')
    print(report)
    report_path = phase_dir / 'reporte_clasificacion.txt'
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(report)
    print(f'Reporte guardado en: {report_path}')

    show_sample_predictions(model, val_dataset, class_names, device, phase_dir, 'Validación')
    evaluate_on_test(model, class_names, val_transform, batch_size, num_workers, device, phase_dir)

    return model, history, best_val_acc

In [ ]:
# ==================================================
# 6) Definición de épocas por fase (N, M, K)
# ==================================================
N = 20  # Fase 1
M = 15  # Fase 2
K = 15  # Fase 3

results_root = Path('../resultados/efficientnet_b2_finetuning')
results_root.mkdir(parents=True, exist_ok=True)

print(f'N (fase 1): {N}')
print(f'M (fase 2): {M}')
print(f'K (fase 3): {K}')
print(f'Resultados en: {results_root.resolve()}')

In [ ]:
# ==================================================
# 7) FASE 1 - Backbone congelado, entrenar classifier
# ==================================================
model = create_model(num_classes=num_classes).to(device)
freeze_backbone(model)

phase1_dir = results_root / 'fase_1'
phase1_dir.mkdir(parents=True, exist_ok=True)

model, history_fase1, best_acc_fase1 = train_phase(
    model=model,
    phase_name='FASE_1',
    num_epochs=N,
    lr=1e-3,
    phase_dir=phase1_dir
)

fase1_model_path = Path('../src/modelo_fase1_eficientNet.pth')
torch.save(
    {
        'model_state_dict': model.state_dict(),
        'class_names': class_names,
        'best_val_acc': best_acc_fase1,
        'history': history_fase1
    },
    fase1_model_path
)
print(f'Modelo fase 1 guardado en: {fase1_model_path.resolve()}')

In [ ]:
# ==================================================
# 8) FASE 2 - Descongelar features.8 + classifier
# ==================================================
unfreeze_layers(model, layer_names=['features.8'])

phase2_dir = results_root / 'fase_2'
phase2_dir.mkdir(parents=True, exist_ok=True)

model, history_fase2, best_acc_fase2 = train_phase(
    model=model,
    phase_name='FASE_2',
    num_epochs=M,
    lr=1e-5,
    phase_dir=phase2_dir
)

fase2_model_path = Path('../src/modelo_fase2_eficientNet.pth')
torch.save(
    {
        'model_state_dict': model.state_dict(),
        'class_names': class_names,
        'best_val_acc': best_acc_fase2,
        'history': history_fase2
    },
    fase2_model_path
)
print(f'Modelo fase 2 guardado en: {fase2_model_path.resolve()}')

In [ ]:
# ==================================================
# 9) FASE 3 - Descongelar features.7 + features.8 + classifier
# ==================================================
unfreeze_layers(model, layer_names=['features.7', 'features.8'])

phase3_dir = results_root / 'fase_3'
phase3_dir.mkdir(parents=True, exist_ok=True)

model, history_fase3, best_acc_fase3 = train_phase(
    model=model,
    phase_name='FASE_3',
    num_epochs=K,
    lr=5e-6,
    phase_dir=phase3_dir
)

fase3_model_path = Path('../src/modelo_fase3_eficientNet.pth')
torch.save(
    {
        'model_state_dict': model.state_dict(),
        'class_names': class_names,
        'best_val_acc': best_acc_fase3,
        'history': history_fase3
    },
    fase3_model_path
)
print(f'Modelo fase 3 guardado en: {fase3_model_path.resolve()}')

In [ ]:
# ==================================================
# 10) Resumen final
# ==================================================
print('Entrenamiento progresivo completado.')
print(f'Mejor Val Acc Fase 1: {best_acc_fase1:.4f}')
print(f'Mejor Val Acc Fase 2: {best_acc_fase2:.4f}')
print(f'Mejor Val Acc Fase 3: {best_acc_fase3:.4f}')
print('Modelos guardados:')
print(f'- {fase1_model_path.resolve()}')
print(f'- {fase2_model_path.resolve()}')
print(f'- {fase3_model_path.resolve()}')
print(f'Resultados y gráficas en: {results_root.resolve()}')